# Simulated Annealing for the Travelling Salesman Problem (TSP)

Given **N cities** with Euclidean coordinates $\mathbf{c}_1, \dots, \mathbf{c}_N \in \mathbb{R}^2$, we want a permutation (tour) $\pi$ visiting each city exactly once and returning to the start that minimizes the total length:

$$
L(\pi) = \sum_{k=1}^{N} d(\mathbf{c}_{\pi(k)}, \mathbf{c}_{\pi(k+1)}), \quad \mathbf{c}_{\pi(N+1)} \equiv \mathbf{c}_{\pi(1)}
$$

where $d(\cdot, \cdot)$ is the Euclidean distance. We treat the tour length as **energy**:

$$ E(\pi) = L(\pi) $$

We will use **Simulated Annealing (SA)** to find a near-optimal tour.

In [ ]:
import numpy as np
import random
import matplotlib.pyplot as plt
import time

## 2. Distance and Tour Length

Euclidean distance between two cities $c_i = (x_i, y_i)$ and $c_j = (x_j, y_j)$:

$$ d(c_i, c_j) = \sqrt{(x_i - x_j)^2 + (y_i - y_j)^2} $$

Tour length for a permutation $\pi$:

$$ L(\pi) = \sum_{k=1}^{N} d(\mathbf{c}_{\pi(k)}, \mathbf{c}_{\pi(k+1)}) $$

In [ ]:
def euclidean_distance(c1, c2):
    return np.linalg.norm(np.array(c1) - np.array(c2))

def tour_length(tour, cities):
    L = 0
    N = len(tour)
    for i in range(N):
        L += euclidean_distance(cities[tour[i]], cities[tour[(i+1)%N]])
    return L

## 3. Neighborhood Operators

1. **Swap:** Swap two random cities.
2. **2-opt:** Reverse a sub-segment of the tour.

In [ ]:
def swap_operator(tour):
    i, j = random.sample(range(len(tour)), 2)
    tour_new = tour.copy()
    tour_new[i], tour_new[j] = tour_new[j], tour_new[i]
    return tour_new

def two_opt_operator(tour):
    i, j = sorted(random.sample(range(len(tour)), 2))
    tour_new = tour.copy()
    tour_new[i:j+1] = tour_new[i:j+1][::-1]
    return tour_new

## 4. Initial Solutions

- Random permutation
- Greedy nearest neighbor

In [ ]:
def random_initial(N):
    tour = list(range(N))
    random.shuffle(tour)
    return tour

def greedy_nearest_neighbor(cities):
    N = len(cities)
    unvisited = set(range(N))
    tour = [0]
    unvisited.remove(0)
    while unvisited:
        last = tour[-1]
        next_city = min(unvisited, key=lambda x: euclidean_distance(cities[last], cities[x]))
        tour.append(next_city)
        unvisited.remove(next_city)
    return tour

## 5. Simulated Annealing Algorithm

Metropolis acceptance criterion:

$$
P(\text{accept}) = \begin{cases} 1, & \Delta E \le 0 \\ \exp(-\Delta E / T), & \Delta E > 0 \end{cases}
$$

Cooling schedules:
- Geometric: $T_{k+1} = \alpha T_k$
- Linear: $T_{k+1} = T_k - \beta$
- Logarithmic: $T_k = \frac{T_0}{\log(1+k)}$

In [ ]:
def simulated_annealing(cities, initial_tour, neighbour_op='swap', 
                        T0=1000, Tmin=1e-3, alpha=0.995, L_iter=None, 
                        schedule='geometric', max_iter=10000):
    N = len(cities)
    if L_iter is None:
        L_iter = 10 * N

    current_tour = initial_tour.copy()
    current_energy = tour_length(current_tour, cities)
    best_tour = current_tour.copy()
    best_energy = current_energy

    energy_history = [current_energy]
    T = T0
    iter_count = 0

    while T > Tmin and iter_count < max_iter:
        for _ in range(L_iter):
            if neighbour_op == 'swap':
                new_tour = swap_operator(current_tour)
            elif neighbour_op == '2-opt':
                new_tour = two_opt_operator(current_tour)
            else:
                raise ValueError("Invalid neighbour operator")
            
            new_energy = tour_length(new_tour, cities)
            delta = new_energy - current_energy
            
            if delta < 0 or random.random() < np.exp(-delta/T):
                current_tour = new_tour
                current_energy = new_energy
                if current_energy < best_energy:
                    best_tour = current_tour.copy()
                    best_energy = current_energy
        
        energy_history.append(best_energy)
        
        if schedule == 'geometric':
            T *= alpha
        elif schedule == 'linear':
            T -= alpha
        elif schedule == 'logarithmic':
            iter_count += 1
            T = T0 / np.log(1 + iter_count)
        else:
            raise ValueError("Invalid cooling schedule")
        iter_count += 1

    return best_tour, best_energy, energy_history

## 6. Example Usage and Visualization

In [ ]:
N = 10
np.random.seed(42)
cities = np.random.rand(N, 2) * 100

# Initial tours
initial_random = random_initial(N)
initial_greedy = greedy_nearest_neighbor(cities)

# Run SA
best_tour, best_energy, history = simulated_annealing(
    cities, initial_random, neighbour_op='swap', T0=1000, Tmin=1e-3, alpha=0.995
)

best_tour_2opt, best_energy_2opt, history_2opt = simulated_annealing(
    cities, initial_greedy, neighbour_op='2-opt', T0=1000, Tmin=1e-3, alpha=0.995
)

print("Best tour (swap, random init):", best_tour)
print("Tour length:", best_energy)
print("Best tour (2-opt, greedy init):", best_tour_2opt)
print("Tour length:", best_energy_2opt)

# Convergence plot
plt.plot(history, label='Swap, Random Init')
plt.plot(history_2opt, label='2-opt, Greedy Init')
plt.xlabel('Iteration')
plt.ylabel('Tour Length')
plt.title('SA Convergence')
plt.legend()
plt.show()

## 7. Visualize Best Tour

In [ ]:
def plot_tour(cities, tour, title="TSP Tour"):
    tour_coords = cities[tour + [tour[0]]]
    plt.figure(figsize=(6,6))
    plt.plot(tour_coords[:,0], tour_coords[:,1], 'o-', markersize=10)
    for i, (x, y) in enumerate(cities):
        plt.text(x+0.5, y+0.5, str(i))
    plt.title(title)
    plt.xlabel("X")
    plt.ylabel("Y")
    plt.show()

plot_tour(cities, best_tour_2opt, "Best Tour (2-opt, Greedy Init)")